# EUVP CycleGAN 推理与评估笔记
本笔记在项目根目录下运行。建议：
- 使用预训练或自有权重在 EUVP 数据集上进行推理（单目录）。
- 结果输出至 `results/`。
- 需要安装 `opencv-contrib-python` 以计算 PSNR/SSIM。

如需使用你在 EUVP 上训练的模型，将 `CHECKPOINTS_NAME` 改为 `euvp_cyclegan`，并确保 `checkpoints/euvp_cyclegan/` 下存在 `latest_net_G_A.pth` 或 `latest_net_G_B.pth`。


In [1]:
# 鍩烘湰璺緞涓庡弬鏁伴厤缃甛n
import sys, os
from pathlib import Path

REPO_ROOT = Path(r'd:\VScode\Graduation project\pytorch-CycleGAN-and-pix2pix-master\pytorch-CycleGAN-and-pix2pix-master')
DATA_INP = Path(r'd:\VScode\Graduation project\EUVP Dataset\test_samples\Inp')
DATA_GTR = Path(r'd:\VScode\Graduation project\EUVP Dataset\test_samples\GTr')

# 棰勮缁冩紨绀猴細horse2zebra锛涜嫢鐢ㄤ綘鑷繁鐨凟U VP璁粌鏉冮噸锛屾敼涓?euvp_cyclegan'
CHECKPOINTS_NAME = 'horse2zebra_pretrained'  # 鎴?'euvp_cyclegan'
EPOCH = 'latest'
GPU_IDS = '0'  # '-1'琛ㄧずCPU锛?0'琛ㄧず浣跨敤GPU 0
NUM_TEST = 20     # 鍏堣窇涓€閮ㄥ垎锛屼究浜庡揩閫熼瑙圽n
NETG = 'resnet_9blocks'
NORM = 'instance'

assert REPO_ROOT.exists(), f'椤圭洰鏍硅矾寰勪笉瀛樺湪: {REPO_ROOT}'
assert DATA_INP.exists(), f'杈撳叆鐩綍涓嶅瓨鍦? {DATA_INP}'
print('Repo:', REPO_ROOT)
print('Input dir:', DATA_INP)
print('GT dir:', DATA_GTR)
print('Checkpoints name:', CHECKPOINTS_NAME)


Repo: d:\VScode\Graduation project\pytorch-CycleGAN-and-pix2pix-master\pytorch-CycleGAN-and-pix2pix-master
Input dir: d:\VScode\Graduation project\EUVP Dataset\test_samples\Inp
GT dir: d:\VScode\Graduation project\EUVP Dataset\test_samples\GTr
Checkpoints name: horse2zebra_pretrained


In [2]:
# 锛堝彲閫夛級涓嬭浇棰勮缁冪敓鎴愬櫒鍒伴」鐩?checkpoints
import urllib.request, time
ckpt_dir = REPO_ROOT / 'checkpoints' / CHECKPOINTS_NAME
ckpt_dir.mkdir(parents=True, exist_ok=True)

def download_with_retry(url, dst, retries=3, min_bytes=10000000):
    for i in range(retries):
        tmp = dst.with_suffix('.part')
        try:
            print(f'[Attempt {i+1}/{retries}] Downloading:', url)
            urllib.request.urlretrieve(url, tmp)
            size = tmp.stat().st_size
            print('Downloaded bytes:', size)
            if size < min_bytes:
                print('File too small; retrying...')
                tmp.unlink(missing_ok=True)
                time.sleep(1)
                continue
            tmp.replace(dst)
            return True
        except Exception as e:
            print('Download error:', e)
            time.sleep(1)
    return False

if CHECKPOINTS_NAME.endswith('_pretrained'):
    target = ckpt_dir / 'latest_net_G.pth'
    url = 'http://efrosgans.eecs.berkeley.edu/cyclegan/pretrained_models/horse2zebra.pth'
    need_download = True
    if target.exists():
        size = target.stat().st_size
        if size >= 40000000:
            print('Checkpoint exists, size:', size)
            need_download = False
        else:
            print('Existing checkpoint seems truncated; re-downloading.')
            target.unlink(missing_ok=True)
    if need_download:
        ok = download_with_retry(url, target, retries=3, min_bytes=40000000)
        if not ok:
            raise RuntimeError('棰勮缁冩潈閲嶄笅杞藉け璐ワ紝璇锋鏌ョ綉缁滄垨绋嶅悗閲嶈瘯銆?')
        print('Saved to:', target)
else:
    print('浣跨敤鑷湁鏉冮噸锛屽亣璁炬鏌ョ偣浣嶄簬: ', ckpt_dir)


Checkpoint exists, size: 45575747


In [3]:
# 妫€娴嬪彲鐢ㄧ殑妫€鏌ョ偣鏂囦欢骞跺喅瀹?model_suffix
ckpt_A = ckpt_dir / f'{EPOCH}_net_G_A.pth'
ckpt_B = ckpt_dir / f'{EPOCH}_net_G_B.pth'
ckpt_G = ckpt_dir / f'{EPOCH}_net_G.pth'
model_suffix = ''
if ckpt_A.exists():
    model_suffix = '_A'
elif ckpt_B.exists():
    model_suffix = '_B'
elif ckpt_G.exists():
    model_suffix = ''
else:
    raise FileNotFoundError(f'鏈壘鍒版鏌ョ偣: {ckpt_dir} 涓嬬殑 {EPOCH}_net_G(_A/_B).pth')
print('浣跨敤 model_suffix:', model_suffix or '(none)')


浣跨敤 model_suffix: (none)


In [4]:
# 杩愯鍗曠洰褰曟帹鐞嗭細--model test --dataset_mode single
import subprocess, shlex, os
# 纭繚 dominate 渚濊禆鍙敤锛坲til/html 闇€瑕侊級
try:
    import dominate
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dominate'], check=True)
# 纭繚 wandb 鍙敤锛坴isualizer 瀵煎叆闇€瑕侊級锛涘悓鏃剁鐢ㄥ叾鑱旂綉涓庢棩蹇梊n
try:
    import wandb
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
os.environ['WANDB_DISABLED'] = 'true'
cmd = [
    sys.executable, str(REPO_ROOT / 'test.py'),
    '--dataroot', str(DATA_INP),
    '--name', CHECKPOINTS_NAME,
    '--model', 'test',
    '--dataset_mode', 'single',
    '--netG', NETG,
    '--norm', NORM,
    '--epoch', EPOCH,
    '--num_test', str(NUM_TEST),
]
if model_suffix:
    cmd += ['--model_suffix', model_suffix]
# 缁熶竴鐢?CUDA_VISIBLE_DEVICES 鎺у埗璁惧锛欳PU 璁句负绌哄瓧绗︿覆锛孏PU 璁惧叿浣撶紪鍙穃n
env = os.environ.copy()
if GPU_IDS.strip() == '-1':
    env['CUDA_VISIBLE_DEVICES'] = ''
else:
    env['CUDA_VISIBLE_DEVICES'] = GPU_IDS
print('Running:', ' '.join(shlex.quote(c) for c in cmd))
proc = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env, capture_output=True, text=True)
print('Return code:', proc.returncode)
if proc.stdout:
    print('--- stdout ---')
    print(proc.stdout)
if proc.stderr:
    print('--- stderr ---')
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError('鎺ㄧ悊澶辫触锛氳妫€鏌ヤ笂鏂?stderr/stdout 涓庝緷璧栭」銆?')
print('Done.')


Running: 'c:\Users\86157\anaconda3\envs\pytorch\python.exe' 'd:\VScode\Graduation project\pytorch-CycleGAN-and-pix2pix-master\pytorch-CycleGAN-and-pix2pix-master\test.py' --dataroot 'd:\VScode\Graduation project\EUVP Dataset\test_samples\Inp' --name horse2zebra_pretrained --model test --dataset_mode single --netG resnet_9blocks --norm instance --epoch latest --num_test 20
Return code: 0
--- stdout ---
----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: d:\VScode\Graduation project\EUVP Dataset\test_samples\Inp	[default: None]
             dataset_mode: single                        
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest            

In [5]:
# 棰勮閮ㄥ垎鐢熸垚缁撴灉
from IPython.display import Image, display
results_dir = REPO_ROOT / 'results' / CHECKPOINTS_NAME / f'test_{EPOCH}' / 'images'
print('Results dir:', results_dir)
generated = sorted(results_dir.glob('*_fake_*.png'))[:3]
if not generated:
    print('鏆傛棤鍖归厤鐢熸垚鏂囦欢锛岃妫€鏌ユ帹鐞嗘槸鍚︽垚鍔熴€?')
else:
    for p in generated:
        print(p.name)
        display(Image(filename=str(p)))


Results dir: d:\VScode\Graduation project\pytorch-CycleGAN-and-pix2pix-master\pytorch-CycleGAN-and-pix2pix-master\results\horse2zebra_pretrained\test_latest\images
鏆傛棤鍖归厤鐢熸垚鏂囦欢锛岃妫€鏌ユ帹鐞嗘槸鍚︽垚鍔熴€?


In [6]:
# 渚濊禆瀹夎锛堣嫢缂哄け鍒欏畨瑁咃級
try:
    import cv2, numpy as np
except Exception as e:
    import sys
    print('瀹夎渚濊禆: opencv-contrib-python numpy')
    !{sys.executable} -m pip install -q opencv-contrib-python numpy
    import cv2, numpy as np
print('OpenCV:', cv2.__version__)


OpenCV: 4.9.0


In [7]:
# 璁＄畻 PSNR/SSIM锛堝熀浜?EUVP 娴嬭瘯鏍锋湰锛塡n
import cv2, numpy as np
def compute_psnr(a, b):
    return cv2.PSNR(a, b)
def compute_ssim(a, b):
    return cv2.quality.QualitySSIM_compute(a, b)[0].mean()

inp_files = sorted([p for p in DATA_INP.glob('**/*') if p.is_file()])
psnrs, ssims = [], []
matched = 0
for inp_path in inp_files:
    stem = inp_path.stem
    candidates = list(results_dir.glob(f'{stem}*_fake_*.png')) or list(results_dir.glob(f'{stem}*.jpg'))
    if not candidates:
        continue
    pred_path = candidates[0]
    gtr_candidates = list(DATA_GTR.glob(f'{stem}*'))
    if not gtr_candidates:
        continue
    gtr_path = gtr_candidates[0]

    inp_img = cv2.imread(str(inp_path), cv2.IMREAD_COLOR)
    pred_img = cv2.imread(str(pred_path), cv2.IMREAD_COLOR)
    gtr_img = cv2.imread(str(gtr_path), cv2.IMREAD_COLOR)

    h, w = inp_img.shape[:2]
    if pred_img.shape[:2] != (h, w):
        pred_img = cv2.resize(pred_img, (w, h), interpolation=cv2.INTER_AREA)
    if gtr_img.shape[:2] != (h, w):
        gtr_img = cv2.resize(gtr_img, (w, h), interpolation=cv2.INTER_AREA)

    psnrs.append(compute_psnr(gtr_img, pred_img))
    ssims.append(compute_ssim(gtr_img, pred_img))
    matched += 1

print('Matched pairs:', matched)
if matched:
    print(f'Average PSNR: {np.mean(psnrs):.4f} dB')
    print(f'Average SSIM: {np.mean(ssims):.4f}')
else:
    print('No matches; 璇锋鏌ョ敓鎴愭枃浠跺悕涓庣洰褰曘€?')


Matched pairs: 0
No matches; 璇锋鏌ョ敓鎴愭枃浠跺悕涓庣洰褰曘€?


## 说明与提示
- 如果使用你在 EUVP 上训练的模型，请设置 `CHECKPOINTS_NAME='euvp_cyclegan'`，并确保 `checkpoints/euvp_cyclegan/` 下有 `latest_net_G_A.pth` 或 `latest_net_G_B.pth`。
- 若 `validation` 仅包含单域图片，建议使用单目录测试（本笔记已采用）。
- 输出结果位于 `results/<name>/test_latest/images/`，可在本笔记中直接预览生成图。
- PSNR/SSIM 依赖 `opencv-contrib-python`，如计算失败请检查是否已正确安装。
